In [1]:
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")


def call_model(state: MessagesState):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

In [2]:
builder = StateGraph(MessagesState)

builder.add_node("chat", call_model)
builder.add_edge(START,"chat")
builder.add_edge("chat",END)

In [3]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable":{"thread_id":"thread-1"}}

graph.invoke({"messages":["My name is Aayush Aswal"]},config=config)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'messages': [HumanMessage(content='My name is Aayush Aswal', additional_kwargs={}, response_metadata={}, id='19faa2d3-9a80-4237-90a4-cd221c65e515'),
  AIMessage(content=[{'type': 'text', 'text': "Hello Aayush! It's great to meet you. How can I help you today?", 'extras': {'signature': 'Eq8FCqwFAWkUfROG7sqhfEzcVLy8mZcYA0TBOd9kyppw7sAuMT5n04qgSVosooeDH9PjX4dQKiZ7d9jHEp0vkqH9c+CCUcTRWPa0xGG90C7wZuQ1uu+0lK4/HueL6WbN8eLKEth10fEasQ7VrzKtR6iiNunoWReECN0Z6BUdd1kvuhR0yAvohp5SX00tHFBpu1SfjVbg3G9a4MzDz0Da+H5Hsc+uX8La6j6I4kH4GA6Vrnat7xydmEsDDikUTbw9X8GQroaovJMQpZrOdbKLKg40MKSIWDxjjm59kFPv5PR063qM1d9kCoVkVaV8UyJgtWkDXW6txM4jPoiqFgX24jE49sh0FLu5DzLlWswo7Q+blcZ7njyTAGrY21IYW/941vqzQkrT15ym5Apq/bFgMYxhh9kJmfA4ePQbtxmNYtM+qiq2h4gYccZtJeZbZQ7fOSEPG/M5VfTC+E5qh0625/eQvXP5an3qlQ7F8xi/RBC48W1W7gMr67/Tv16mlqDki5F9bw2INzVE6MJidI/ol/zksyQ4W6jK1WdOmMF+bIyIDWPwWJBNxueerCrOuolws/82y93CHV/u36YC2wjZRWPjmyYQduCFXpXHml/vZK6V6twM4FwrvEIovnsrNnQU7UkThq7j70I41lpp0kJFokg5A/2Iulx0GZYmKtvKAp1Vqoc9jZpotj5ZXsz5HEC6HuNcQQGe

In [4]:
graph.invoke({"messages":["What is My Name?"]},config=config)

{'messages': [HumanMessage(content='My name is Aayush Aswal', additional_kwargs={}, response_metadata={}, id='19faa2d3-9a80-4237-90a4-cd221c65e515'),
  AIMessage(content=[{'type': 'text', 'text': "Hello Aayush! It's great to meet you. How can I help you today?", 'extras': {'signature': 'Eq8FCqwFAWkUfROG7sqhfEzcVLy8mZcYA0TBOd9kyppw7sAuMT5n04qgSVosooeDH9PjX4dQKiZ7d9jHEp0vkqH9c+CCUcTRWPa0xGG90C7wZuQ1uu+0lK4/HueL6WbN8eLKEth10fEasQ7VrzKtR6iiNunoWReECN0Z6BUdd1kvuhR0yAvohp5SX00tHFBpu1SfjVbg3G9a4MzDz0Da+H5Hsc+uX8La6j6I4kH4GA6Vrnat7xydmEsDDikUTbw9X8GQroaovJMQpZrOdbKLKg40MKSIWDxjjm59kFPv5PR063qM1d9kCoVkVaV8UyJgtWkDXW6txM4jPoiqFgX24jE49sh0FLu5DzLlWswo7Q+blcZ7njyTAGrY21IYW/941vqzQkrT15ym5Apq/bFgMYxhh9kJmfA4ePQbtxmNYtM+qiq2h4gYccZtJeZbZQ7fOSEPG/M5VfTC+E5qh0625/eQvXP5an3qlQ7F8xi/RBC48W1W7gMr67/Tv16mlqDki5F9bw2INzVE6MJidI/ol/zksyQ4W6jK1WdOmMF+bIyIDWPwWJBNxueerCrOuolws/82y93CHV/u36YC2wjZRWPjmyYQduCFXpXHml/vZK6V6twM4FwrvEIovnsrNnQU7UkThq7j70I41lpp0kJFokg5A/2Iulx0GZYmKtvKAp1Vqoc9jZpotj5ZXsz5HEC6HuNcQQGe

In [6]:
snap = graph.get_state(config)
vals = snap.values
for m in vals.get("messages",[]):
    print("-",type(m).__name__,":",m.content)

- HumanMessage : My name is Aayush Aswal
- AIMessage : [{'type': 'text', 'text': "Hello Aayush! It's great to meet you. How can I help you today?", 'extras': {'signature': 'Eq8FCqwFAWkUfROG7sqhfEzcVLy8mZcYA0TBOd9kyppw7sAuMT5n04qgSVosooeDH9PjX4dQKiZ7d9jHEp0vkqH9c+CCUcTRWPa0xGG90C7wZuQ1uu+0lK4/HueL6WbN8eLKEth10fEasQ7VrzKtR6iiNunoWReECN0Z6BUdd1kvuhR0yAvohp5SX00tHFBpu1SfjVbg3G9a4MzDz0Da+H5Hsc+uX8La6j6I4kH4GA6Vrnat7xydmEsDDikUTbw9X8GQroaovJMQpZrOdbKLKg40MKSIWDxjjm59kFPv5PR063qM1d9kCoVkVaV8UyJgtWkDXW6txM4jPoiqFgX24jE49sh0FLu5DzLlWswo7Q+blcZ7njyTAGrY21IYW/941vqzQkrT15ym5Apq/bFgMYxhh9kJmfA4ePQbtxmNYtM+qiq2h4gYccZtJeZbZQ7fOSEPG/M5VfTC+E5qh0625/eQvXP5an3qlQ7F8xi/RBC48W1W7gMr67/Tv16mlqDki5F9bw2INzVE6MJidI/ol/zksyQ4W6jK1WdOmMF+bIyIDWPwWJBNxueerCrOuolws/82y93CHV/u36YC2wjZRWPjmyYQduCFXpXHml/vZK6V6twM4FwrvEIovnsrNnQU7UkThq7j70I41lpp0kJFokg5A/2Iulx0GZYmKtvKAp1Vqoc9jZpotj5ZXsz5HEC6HuNcQQGeC8GOaHBhPVdPZay0DYigy4MK4pLcggN2ZLcvXxOfEw1p/CXav0Zv/4tjLo4+xHPlrn3NTw2bExMQdf/XLa3sREKMXT+Jtybeb5kjpDHtRv4cxv13OjB